# 如何使用聊天模型调用函数

本 notebook 介绍如何将 Chat Completions API 与外部函数结合使用，以扩展 GPT 模型的能力。

`tools` 是 Chat Completion API 中的一个可选参数，可用于提供函数规范。其目的是让模型能够生成符合所提供规范的函数参数。请注意，API 实际上不会执行任何函数调用，需要由开发者根据模型输出来执行函数调用。

在 `tools` 参数中，如果提供了 `functions` 参数，那么默认情况下模型会自行决定何时适合使用其中某个函数。可以通过将 `tool_choice` 参数设置为 `{"type": "function", "function": {"name": "my_function"}}` 来强制 API 使用特定函数。也可以通过将 `tool_choice` 参数设置为 `"none"` 来强制 API 不使用任何函数。如果使用了函数，输出的响应中将包含 `"finish_reason": "tool_calls"`，以及一个包含函数名称和生成的函数参数的 `tool_calls` 对象。

### 概览

本 notebook 包含以下 2 个部分：

- **如何生成函数参数：** 指定一组函数，并使用 API 生成函数参数。
- **如何使用模型生成的参数调用函数：** 使用模型生成的参数实际执行函数，形成完整闭环。

## 如何生成函数参数

In [1]:
!pip install scipy --quiet
!pip install tenacity --quiet
!pip install tiktoken --quiet
!pip install termcolor --quiet
!pip install openai --quiet

In [2]:
import json
from openai import OpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored  

GPT_MODEL = "gpt-5"
client = OpenAI()

### 工具函数

首先，我们定义几个工具函数，用于调用 Chat Completions API 以及维护和跟踪对话状态。

In [3]:
@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, tools=None, tool_choice=None, model=GPT_MODEL):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice,
        )
        return response
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e


In [4]:
def pretty_print_conversation(messages):
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "function": "magenta",
    }
    
    for message in messages:
        if message["role"] == "system":
            print(colored(f"system: {message['content']}\n", role_to_color[message["role"]]))
        elif message["role"] == "user":
            print(colored(f"user: {message['content']}\n", role_to_color[message["role"]]))
        elif message["role"] == "assistant" and message.get("tool_calls"):
            print(colored(f"assistant: {message['tool_calls']}\n", role_to_color[message["role"]]))
        elif message["role"] == "assistant" and not message.get("tool_calls"):
            print(colored(f"assistant: {message['content']}\n", role_to_color[message["role"]]))
        elif message["role"] == "function":
            print(colored(f"function ({message['name']}): {message['content']}\n", role_to_color[message["role"]]))


### 基本概念

我们来创建一些函数规范，用于对接一个假想的天气 API。我们会将这些函数规范传给 Chat Completions API，以生成符合规范的函数参数。

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this unit from the forecast location.",
                    },
                },
                "required": ["location", "format"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_n_day_weather_forecast",
            "description": "Get an N-day weather forecast",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this unit from the forecast location.",
                    },
                    "num_days": {
                        "type": "integer",
                        "description": "The number of days to forecast",
                    }
                },
                "required": ["location", "format", "num_days"]
            },
        }
    },
]

如果我们向模型询问当前天气，它会回复一些澄清性的问题。

In [6]:
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "What's the weather like today"})
chat_response = chat_completion_request(
    messages, tools=tools
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)


system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: What's the weather like today

assistant: Sure—what city and state (or country) should I check? Also, do you prefer Celsius or Fahrenheit?



当我们提供缺失的信息后，它就会为我们生成合适的函数参数。

In [7]:
messages.append({"role": "user", "content": "I'm in Glasgow, Scotland."})
chat_response = chat_completion_request(
    messages, tools=tools
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)


system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: What's the weather like today

assistant: Sure—what city and state (or country) should I check? Also, do you prefer Celsius or Fahrenheit?

user: I'm in Glasgow, Scotland.

assistant: [{'id': 'call_k2QgGc9GT9WjxD76GvR0Ot8q', 'function': {'arguments': '{"location": "Glasgow, Scotland", "format": "celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}, {'id': 'call_RtnXV5t49lqbWwhvGoEPZ7KY', 'function': {'arguments': '{"location": "Glasgow, Scotland", "format": "celsius", "num_days": 1}', 'name': 'get_n_day_weather_forecast'}, 'type': 'function'}]



通过换一种方式提示，我们可以让它使用我们告诉它的另一个函数。

In [8]:
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "what is the weather going to be like in Glasgow, Scotland over the next x days"})
chat_response = chat_completion_request(
    messages, tools=tools
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)


system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: what is the weather going to be like in Glasgow, Scotland over the next x days

assistant: How many days would you like the forecast for in Glasgow, Scotland? For example: 3, 5, 7, 10, or 14.



模型再次向我们寻求澄清，因为它还没有足够的信息。在这种情况下，它已经知道预报的地点，但还需要知道预报需要多少天。

In [9]:
messages.append({"role": "user", "content": "5 days"})
chat_response = chat_completion_request(
    messages, tools=tools
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)


system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: what is the weather going to be like in Glasgow, Scotland over the next x days

assistant: How many days would you like the forecast for in Glasgow, Scotland? For example: 3, 5, 7, 10, or 14.

user: 5 days

assistant: [{'id': 'call_lNzOVLrNSaSVjL3O3bN110af', 'function': {'arguments': '{"location":"Glasgow, Scotland","format":"celsius","num_days":5}', 'name': 'get_n_day_weather_forecast'}, 'type': 'function'}]



#### 强制使用特定函数或不使用函数

我们可以使用 function_call 参数强制模型使用特定函数，例如 get_n_day_weather_forecast。这样做会迫使模型对如何使用该函数做出假设。

In [10]:
# in this cell we force the model to use get_n_day_weather_forecast
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "Give me a weather report for Toronto, Canada."})
chat_response = chat_completion_request(
    messages, tools=tools, tool_choice={"type": "function", "function": {"name": "get_n_day_weather_forecast"}}
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)

system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: Give me a weather report for Toronto, Canada.

assistant: [{'id': 'call_3hoMjl55OQ7LxfwhFyjxwv1T', 'function': {'arguments': '{"location":"Toronto, Canada","format":"celsius","num_days":5}', 'name': 'get_n_day_weather_forecast'}, 'type': 'function'}]



In [11]:
# if we don't force the model to use get_n_day_weather_forecast it may not
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "Give me a weather report for Toronto, Canada."})
chat_response = chat_completion_request(
    messages, tools=tools
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)

system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: Give me a weather report for Toronto, Canada.

assistant: [{'id': 'call_wv5mdjEQJnBPuSci3xw09Tom', 'function': {'arguments': '{"location":"Toronto, ON","format":"celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}]



我们也可以强制模型完全不使用函数。这样做会阻止它生成正确的函数调用。

In [12]:
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "Give me the current weather (use Celcius) for Toronto, Canada."})
chat_response = chat_completion_request(
    messages, tools=tools, tool_choice="none"
)
messages.append(chat_response.choices[0].message.to_dict())
pretty_print_conversation(messages)

system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: Give me the current weather (use Celcius) for Toronto, Canada.

assistant: I don’t have live access to pull the current conditions right now. To get Toronto’s current weather in Celsius, check any of these quickly:
- Environment Canada: weather.gc.ca (search “Toronto”)
- The Weather Network: theweathernetwork.com/ca/weather/ontario/toronto
- Google: search “Toronto weather” (shows °C by default in Canada)
- AccuWeather or Weather.com (set units to °C)

If you paste the current readings here (temperature, feels-like, wind, precipitation), I can interpret them and advise on what to wear or plan for.



### 并行函数调用

gpt-5、gpt-4.1 或 gpt-4o 等较新的模型可以在一轮中调用多个函数。

In [13]:
messages = []
messages.append({"role": "system", "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."})
messages.append({"role": "user", "content": "what is the weather going to be like in San Francisco and Glasgow over the next 4 days"})
chat_response = chat_completion_request(
    messages, tools=tools, model="gpt-4o"
)

assistant_message = chat_response.choices[0].message.tool_calls
assistant_message

[ChatCompletionMessageFunctionToolCall(id='call_KlZ3Fqt3SviC6o66dVMYSa2Q', function=Function(arguments='{"location": "San Francisco, CA", "format": "fahrenheit", "num_days": 4}', name='get_n_day_weather_forecast'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_YAnH0VRB3oqjqivcGj3Cd8YA', function=Function(arguments='{"location": "Glasgow, UK", "format": "celsius", "num_days": 4}', name='get_n_day_weather_forecast'), type='function')]

## 如何使用模型生成的参数调用函数

在下一个示例中，我们将演示如何执行输入由模型生成的函数，并以此实现一个能够为我们回答数据库相关问题的智能体。为简单起见，我们将使用 [Chinook 示例数据库](https://www.sqlitetutorial.net/sqlite-sample-database/)。

*注意：* 在生产环境中，SQL 生成可能具有较高风险，因为模型在生成正确 SQL 方面并非完全可靠。

### 指定用于执行 SQL 查询的函数

首先，我们定义一些实用的工具函数，用于从 SQLite 数据库中提取数据。

In [14]:
import sqlite3

conn = sqlite3.connect("data/Chinook.db")
print("Opened database successfully")

Opened database successfully


In [15]:
def get_table_names(conn):
    """Return a list of table names."""
    table_names = []
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    for table in tables.fetchall():
        table_names.append(table[0])
    return table_names


def get_column_names(conn, table_name):
    """Return a list of column names."""
    column_names = []
    columns = conn.execute(f"PRAGMA table_info('{table_name}');").fetchall()
    for col in columns:
        column_names.append(col[1])
    return column_names


def get_database_info(conn):
    """Return a list of dicts containing the table name and columns for each table in the database."""
    table_dicts = []
    for table_name in get_table_names(conn):
        columns_names = get_column_names(conn, table_name)
        table_dicts.append({"table_name": table_name, "column_names": columns_names})
    return table_dicts


现在我们可以使用这些工具函数来提取数据库 schema 的表示。

In [16]:
database_schema_dict = get_database_info(conn)
database_schema_string = "\n".join(
    [
        f"Table: {table['table_name']}\nColumns: {', '.join(table['column_names'])}"
        for table in database_schema_dict
    ]
)

与之前一样，我们为希望 API 生成参数的函数定义一个函数规范。注意，我们将数据库 schema 插入到了函数规范中。这对模型了解数据库结构非常重要。

In [17]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "ask_database",
            "description": "Use this function to answer user questions about music. Input should be a fully formed SQL query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": f"""
                                SQL query extracting info to answer the user's question.
                                SQL should be written using this database schema:
                                {database_schema_string}
                                The query should be returned in plain text, not in JSON.
                                """,
                    }
                },
                "required": ["query"],
            },
        }
    }
]

### 执行 SQL 查询

现在我们来实现真正对数据库执行查询的函数。

In [18]:
def ask_database(conn, query):
    """Function to query SQLite database with a provided SQL query."""
    try:
        results = str(conn.execute(query).fetchall())
    except Exception as e:
        results = f"query failed with error: {e}"
    return results

##### 使用 Chat Completions API 发起函数调用的步骤：

**第 1 步**：用可能促使模型选择工具的内容提示模型。工具的描述（如函数名称和签名）定义在 'Tools' 列表中，并在 API 调用中传给模型。如果模型选择了工具，响应中会包含函数名称和参数。<br>
  
**第 2 步**：以编程方式检查模型是否想要调用函数。如果是，则继续第 3 步。<br>  
**第 3 步**：从响应中提取函数名称和参数，使用这些参数调用函数，并将结果追加到 messages 中。<br>    
**第 4 步**：使用消息列表调用 chat completions API 以获取响应。

In [21]:
# Step #1: Prompt with content that may result in function call. In this case the model can identify the information requested by the user is potentially available in the database schema passed to the model in Tools description. 
messages = [{
    "role":"user", 
    "content": "What is the name of the album with the most tracks?"
}]

response = client.chat.completions.create(
    model=GPT_MODEL, 
    messages=messages, 
    tools=tools, 
    tool_choice="auto"
)

# Append the message to messages list
response_message = response.choices[0].message 
messages.append(response_message.to_dict())
pretty_print_conversation(messages)

user: What is the name of the album with the most tracks?

assistant: [{'id': 'call_pGRtZZGfd2o41GHlZcEdB9he', 'function': {'arguments': '{"query":"WITH track_counts AS (\\n  SELECT a.AlbumId, a.Title, COUNT(t.TrackId) AS track_count\\n  FROM Album a\\n  JOIN Track t ON t.AlbumId = a.AlbumId\\n  GROUP BY a.AlbumId, a.Title\\n)\\nSELECT Title\\nFROM track_counts\\nWHERE track_count = (SELECT MAX(track_count) FROM track_counts);"}', 'name': 'ask_database'}, 'type': 'function'}]



In [22]:
# Step 2: determine if the response from the model includes a tool call.   
tool_calls = response_message.tool_calls
if tool_calls:
    # If true the model will return the name of the tool / function to call and the argument(s)  
    tool_call_id = tool_calls[0].id
    tool_function_name = tool_calls[0].function.name
    tool_query_string = json.loads(tool_calls[0].function.arguments)['query']

    # Step 3: Call the function and retrieve results. Append the results to the messages list.      
    if tool_function_name == 'ask_database':
        results = ask_database(conn, tool_query_string)
        
        messages.append({
            "role":"tool", 
            "tool_call_id":tool_call_id, 
            "name": tool_function_name, 
            "content":results
        })
        
        # Step 4: Invoke the chat completions API with the function response appended to the messages list
        # Note that messages with role 'tool' must be a response to a preceding message with 'tool_calls'
        model_response_with_function_call = client.chat.completions.create(
            model=GPT_MODEL,
            messages=messages,
        )  # get a new response from the model where it can see the function response
        print(f"Result found in database: {model_response_with_function_call.choices[0].message.content}")
    else: 
        print(f"Error: function {tool_function_name} does not exist")
else: 
    # Model did not identify a function to call, result can be returned to the user 
    print(response_message.content) 

Result found in database: Greatest Hits


## 后续步骤

请参阅我们的另一个 [notebook](How_to_call_functions_for_knowledge_retrieval.ipynb)，其中演示了如何使用 Chat Completions API 和函数进行知识检索，以对话方式与知识库交互。